# Day 08 — Chạy model pre-label trên Google Colab (cho máy yếu)

**Dùng khi:** laptop không GPU hoặc quá chậm. Colab cho GPU miễn phí (T4) để **chạy model** (Part 2, 3, 4, 5). Phần **CVAT, review, đo số, viết báo cáo vẫn làm trên máy bạn**.

**Luồng:** Colab chạy `prelabel.py` → tải file `pre.coco.json` về máy → nạp vào CVAT (cách A trong `docs/CVAT_AI.md`) → review → export → các script đo (`review_stats.py`...) chạy trên máy bạn (nhẹ, không cần GPU).

> Colab **không** nối được tới CVAT chạy `localhost` của bạn. Đó là lý do ta chỉ lấy file nhãn từ Colab.
> Bài vẫn là **cá nhân**: commit sản phẩm từ repo trên máy bạn, không phải từ Colab.

**Trước khi chạy:** menu **Runtime → Change runtime type → T4 GPU**. Chạy các ô theo thứ tự từ trên xuống.

## 0. Kiểm tra GPU và lấy mã nguồn

In [ ]:
!nvidia-smi -L || echo "Không có GPU: vào Runtime > Change runtime type > T4 GPU"
import os
REPO_URL = "https://github.com/VinUni-AI20k/K5-L23-Day08-AI-Assisted-Labeling.git"   # có thể đổi thành repo fork của bạn (public)
if not os.path.isdir("lab"):
    !git clone -q $REPO_URL lab
%cd lab
!ls

## 1. Cài thư viện (torch đã có sẵn trên Colab, không cài lại)

In [ ]:
!pip -q install "ultralytics>=8.4.0" opencv-python-headless pyyaml matplotlib "transformers>=5.0" huggingface_hub
!python scripts/check_env.py || true

## 2. Tải trọng số
Chọn **phiên bản SAM** của bạn (SAM là bắt buộc): `sam1` (mobile_sam, nhẹ) hoặc `sam2` (SAM 2.1) — `core` đã gồm SAM 2.1 base.

In [ ]:
!python scripts/download_models.py --set core
# !python scripts/download_models.py --set sam1      # tuỳ chọn: SAM 1 (mobile_sam)
# !python scripts/download_models.py --set hf        # tuỳ chọn: Grounding DINO + OWLv2

### (Tuỳ chọn) SAM 3 trên Colab
Cần đã được cấp quyền tại https://huggingface.co/facebook/sam3. Lưu token vào **Colab Secrets** (biểu tượng chìa khoá bên trái) với tên `HF_TOKEN`, bật *Notebook access*, rồi chạy ô dưới. **Không dán token vào notebook.**

In [ ]:
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# !python scripts/download_models.py --set sam3

## 3. Part 2 — Thử model (ví dụ). Mỗi lần chạy ghi một dòng vào `work/runs.csv`
Sửa lệnh / tham số như trong `docs/MODELS.md`. Xem ảnh kết quả ở ô cuối của phần này.

In [ ]:
!python scripts/prelabel.py --model yolo26 --size m --split pilot --conf 0.25 --out work/prelabel/y26_pilot_c025
!python scripts/prelabel.py --model yoloe --classes three_wheeler --split pilot --conf 0.25 --out work/prelabel/yoloe_pilot
# SAM: chọn phiên bản bằng --sam (mobile_sam.pt = SAM 1, sam2.1_b.pt = SAM 2.1)
!python scripts/prelabel.py --model yolo26 --size m --classes person,motorbike,car --images img_004.jpg img_013.jpg --conf 0.3 --segmenter sam --sam sam2.1_b.pt --out work/prelabel/y26_sam

In [ ]:
from IPython.display import Image, display
display(Image("work/prelabel/y26_pilot_c025/viz/img_011.jpg"))
display(Image("work/prelabel/y26_sam/viz/img_013.jpg"))

## 4. Chạy mọi ảnh ở ngưỡng thấp (nguồn cho Part 3, 4)
Chạy **sau khi** bạn đã gán xong 6 ảnh gold bằng tay (Part 1). Ảnh gold không bao giờ được xem pre-label trước khi gán xong.

In [ ]:
!python scripts/prelabel.py --model yolo26 --size m --split all --conf 0.05 --out work/prelabel/y26_all --no-viz
!python scripts/prelabel.py --model yoloe --classes three_wheeler --split all --conf 0.05 --out work/prelabel/yoloe_all --no-viz

## 5. Part 3 — gộp pre-label cho pilot (ngưỡng 0.10)

In [ ]:
!python scripts/merge_prelabels.py work/prelabel/yoloe_all/pre.coco.json work/prelabel/y26_all/pre.coco.json --conf 0.1 --iou 0.6 --split pilot --out work/prelabel/merged_pilot/pre.coco.json

## 6. Part 4 — gộp batch với ngưỡng của bạn
Sau Part 3 (chạy `threshold_sweep.py` **trên máy bạn**), dán chuỗi ngưỡng vào `THR` rồi chạy. Cài **ảnh bẫy** (`inject_traps.py`) làm trên máy bạn để hạt giống và file đáp án nằm trong repo của bạn — hoặc chạy ở đây rồi tải cả hai file về.

In [ ]:
THR = "person=0.35,bicycle=0.3,motorbike=0.35,car=0.3,bus=0.85,truck=0.4,three_wheeler=0.35"   # <-- thay bằng ngưỡng của BẠN
!python scripts/merge_prelabels.py work/prelabel/yoloe_all/pre.coco.json work/prelabel/y26_all/pre.coco.json --thr "$THR" --conf 0.3 --iou 0.6 --split batch,gold --out work/prelabel/merged_review/pre.coco.json
# MSSV = "22001234"   # thay bằng MSHV của bạn rồi bỏ comment dòng dưới nếu muốn cài bẫy ở Colab
# !python scripts/inject_traps.py --pre work/prelabel/merged_review/pre.coco.json --student $MSSV

## 7. Part 5 — mask cho 8 ảnh seg (SAM do bạn chọn)

In [ ]:
SAM = "sam2.1_b.pt"     # SAM 1: mobile_sam.pt | sam_b.pt ; SAM 2.1: sam2.1_t/s/b/l.pt
!python scripts/prelabel.py --model yolo26 --size m --classes person,bicycle,motorbike,car,bus,truck --split seg --thr "$THR" --conf 0.3 --segmenter sam --sam $SAM --out work/prelabel/seg_y26
!python scripts/prelabel.py --model yoloe --classes three_wheeler --split seg --conf 0.25 --segmenter sam --sam $SAM --out work/prelabel/seg_yoloe
!python scripts/merge_prelabels.py work/prelabel/seg_yoloe/pre.coco.json work/prelabel/seg_y26/pre.coco.json --conf 0.25 --iou 0.6 --split seg --out submission/prelabel/seg_pre.coco.json
!python scripts/coco_to_cvat_xml.py --coco submission/prelabel/seg_pre.coco.json --shape polygon --out work/cvat_import/seg_pre.xml

## 8. Tải kết quả về máy
Gói các file cần nạp vào CVAT và `work/runs.csv` (nhật ký các lần chạy, để copy sang `model_log.csv`). Giải nén vào **repo trên máy bạn**, giữ nguyên đường dẫn.

In [ ]:
!mkdir -p work && cp -n work/runs.csv work/runs_backup.csv 2>/dev/null || true
!zip -rq colab_outputs.zip work/prelabel work/cvat_import work/runs.csv submission/prelabel 2>/dev/null || true
from google.colab import files
files.download("colab_outputs.zip")

**Tiếp theo (trên máy bạn):** giải nén `colab_outputs.zip` vào thư mục repo → làm theo `GUIDE.md` từ bước nạp vào CVAT. Nhớ: ghi `model_log.csv` bằng nhận xét **của bạn** khi nhìn ảnh `viz/`; số trong `runs.csv` do Colab chạy (ghi chú `device = cuda`, thời gian khác máy bạn).